"""
=============================================================================
Tendo Model Monitoring — Monthly Power BI Data Pipeline
=============================================================================
Run this script once per month. It pulls all raw data from BigQuery / GCS, applies every transformation from the original analysis notebook, and writes one CSV per dashboard page into the OUTPUT_DIR folder.

Power BI setup (one-time):
  1. In Power BI Desktop → Get Data → Text/CSV → point to each file in OUTPUT_DIR
  2. Each CSV becomes one table — one table per dashboard page
  3. Refresh monthly: run this script, then hit Refresh in Power BI
  
Output files (one per slide/page): <br><br>
    page02_model_performance_gini_cindex.csv <br>
    page03_oop_rank_order_new_users.csv <br>
    page04_oop_rank_order_existing_users.csv <br>
    page05_attrition_rank_order_new_users.csv <br>
    page06_attrition_rank_order_existing_users.csv <br>
    page07_unit_bad_rate_mom.csv <br>
    page08_peso_bad_rate_mom.csv <br>
    page09_cl_plus_exposure.csv <br>
    page10_new_borrower_exposure.csv <br>
    page11_utilization_trend.csv <br>
=============================================================================
"""

# Import Packages

In [1]:
import os
import pickle
import warnings
from datetime import datetime, date

import numpy as np
import pandas as pd
from google.cloud import bigquery, storage
from sklearn.metrics import roc_auc_score
import re

warnings.filterwarnings("ignore")

pd.options.display.max_columns = None

# CONFIGURATION

In [46]:

# =============================================================================
# — edit these each month if needed
# =============================================================================

In [2]:
# GCP credentials — use ADC JSON or set GOOGLE_APPLICATION_CREDENTIALS env var
CREDENTIALS_PATH = r"C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH

os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

PROJECT_ID = "prj-prod-dataplatform"
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"

CLOUDPATH = "DC/Tendo_Model_Monitoring_Data"
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Output folder — Power BI reads CSVs from here
OUTPUT_DIR = r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data"


In [3]:
# Add the new month here each month
REPORT_PERIODS = {
    "Jun 2025": {"start": "2025-06-01", "end": "2025-06-30"},
    "Jul 2025": {"start": "2025-07-01", "end": "2025-07-31"},
    "Aug 2025": {"start": "2025-08-01", "end": "2025-08-31"},
    "Sep 2025": {"start": "2025-09-01", "end": "2025-09-30"},
    "Oct 2025": {"start": "2025-10-01", "end": "2025-10-31"},
    "Nov 2025": {"start": "2025-11-01", "end": "2025-11-30"},
    "Dec 2025": {"start": "2025-12-01", "end": "2025-12-31"},
    # "Jan 2026": {"start": "2026-01-01", "end": "2026-01-31"},
}


In [4]:
ATTRITION_SCORE_MAP = {
    1: "Very high", 2: "Very high", 3: "Very high",
    4: "High",      5: "High",      6: "High",
    7: "Average",   8: "Average",   9: "Average",
    10: "Low",     11: "Low",      12: "Low",
    15: "Very low",
}

In [5]:
ATTRITION_ORDER = {"Very low": 1, "Low": 2, "Average": 3, "High": 4, "Very high": 5}
OOP_SEGMENT_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}

RUN_MONTH = datetime.now().strftime("%Y-%m")

# HELPERS

In [6]:
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def save_csv(df, filename):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    log(f"  ✓ {filename}  ({len(df):,} rows, {len(df.columns)} cols)")


def generate_bucket_url(filename, bucket_name):
    return f"gs://{bucket_name}/{filename}"


def add_date_ym(df, col_name, new_col_name):
    """Add an integer YYYYMM column from a datetime column."""
    dt = pd.to_datetime(df[col_name], errors="coerce")
    df[new_col_name] = dt.dt.year * 100 + dt.dt.month
    return df

def fix_suffix_collision(df, suffix="_x", rename_to=None, drop_suffix="_y"):
    """
    After a merge that produces col_x / col_y pairs, keep _x (left table wins),
    rename it back to the original name, and drop _y.
    rename_to: list of base column names to fix. If None, auto-detects all _x cols.
    """
    if rename_to is None:
        rename_to = [c[:-2] for c in df.columns if c.endswith(suffix)]
    for base in rename_to:
        x_col = base + suffix
        y_col = base + drop_suffix
        if x_col in df.columns:
            df = df.rename(columns={x_col: base})
        if y_col in df.columns:
            df = df.drop(columns=[y_col])
    return df


def save_df_to_gcs(df, bucket_name, destination_blob_name, file_format='csv'):
    """Saves a pandas DataFrame to Google Cloud Storage.

    Args:
        df: The pandas DataFrame to save.
        bucket_name: The name of the GCS bucket.
        destination_blob_name: The name of the blob to be created.
        file_format: The file format to save the DataFrame in ('csv' or 'parquet').
    """

    # Create a temporary file
    if file_format == 'csv':
        temp_file = 'temp.csv'
        df.to_csv(temp_file, index=False)
    elif file_format == 'parquet':
        temp_file = 'temp.parquet'
        df.to_parquet(temp_file, index=False)
    else:
        raise ValueError("Invalid file format. Please choose 'csv' or 'parquet'.")

    # Upload the file to GCS
    storage_client = storage.Client(project="prj-prod-dataplatform")

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    blob.upload_from_filename(temp_file)

    # Remove the temporary file
    import os
    os.remove(temp_file)

import pickle
import io
from google.cloud import storage
def save_pickle_to_gcs(data, bucket_name, destination_blob_name):
    """
    Save any Python object as a pickle file to Google Cloud Storage
    
    Args:
        data: The Python object to pickle (DataFrame, dict, list, etc.)
        bucket_name: Name of the GCS bucket
        destination_blob_name: Path/filename in the bucket
    """
    # Initialize the GCS client
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    
    # Serialize the data to pickle format in memory
    pickle_buffer = io.BytesIO()
    pickle.dump(data, pickle_buffer)
    pickle_buffer.seek(0)
    
    # Upload the pickle data to GCS
    blob.upload_from_file(pickle_buffer, content_type='application/octet-stream')
    print(f"Pickle file uploaded to gs://{bucket_name}/{destination_blob_name}")


def calc_gini(data, date_col, score_col, target_col, periods, weight_col=None):
    """
    Compute Gini for every period window. Returns tidy DataFrame.
    Gini = 2*AUC - 1.  weight_col is optional sample weighting.
    """
    dt = data[data[target_col].notna()].copy()
    dt[date_col] = pd.to_datetime(dt[date_col]).dt.date
    rows = []

    for period_name, info in periods.items():
        start = pd.to_datetime(info["start"]).date()
        end   = pd.to_datetime(info["end"]).date()
        sub   = dt[(dt[date_col] >= start) & (dt[date_col] <= end)].copy()

        n = sub["ee_customer_id"].nunique()
        if n == 0:
            rows.append({"Period": period_name, "Start_Date": start,
                         "End_Date": end, "Sample_Size": 0,
                         "Bad_Rate_Pct": None, "Gini": None})
            continue

        bad_rate = 100 * (
            1 - sub[["ee_customer_id", target_col]]
            .drop_duplicates()[target_col].sum() / n
        )

        gini = None
        if sub[target_col].nunique() >= 2:
            try:
                kw = {"sample_weight": sub[weight_col]} if weight_col else {}
                auc  = roc_auc_score(sub[target_col], sub[score_col], **kw)
                gini = round(2 * auc - 1, 4)
            except Exception as e:
                log(f"    Gini error [{period_name}]: {e}")

        rows.append({
            "Period": period_name, "Start_Date": start, "End_Date": end,
            "Sample_Size": n, "Bad_Rate_Pct": round(bad_rate, 2), "Gini": gini
        })
    return pd.DataFrame(rows)


# =============================================================================

# STEP 1 — LOAD RAW DATA FROM BIGQUERY & GCS

# =============================================================================

In [47]:
## ==============================================================================
## Create Data for Onboarding of user 
## ==============================================================================

In [7]:
client = bigquery.Client(PROJECT_ID)

In [8]:
dt = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard feature data is: {dt.shape}")

dt_repayments = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard repayment data is: {dt_repayments.shape}")




The shape of the tendo scorecard feature data is: (4168379, 100)
The shape of the tendo scorecard repayment data is: (16086906, 15)


## Resignation code

In [9]:
def to_date_str(val) -> str | None:
    # missing
    if val is None or pd.isna(val):
        return None

    # already a string
    if isinstance(val, str):
        return val.strip()

    # datetime-like (Timestamp, datetime, date, numpy datetime64)
    if isinstance(val, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(val).strftime("%Y-%m-%d")

    # fallback (numbers, objects, etc.)
    return str(val).strip()

def validate_and_convert_dates(df, column_name, output_tag_col, output_date_col,
                                min_date='1990-01-01',
                                max_date='2025-10-01',
                                min_date_col=None,
                                max_date_col=None):
    """
    Validates dates and creates tags with converted dates.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with date column
    column_name : str
        Name of the date column to validate
    min_date : str
        Default minimum date threshold (default: '1990-01-01')
    max_date : str
        Default maximum date threshold (default: '2025-10-01')
    min_date_col : str, optional
        Column name with per-row minimum date thresholds (overrides min_date when not null)
    max_date_col : str, optional
        Column name with per-row maximum date thresholds (overrides max_date when not null)

    Tags:
    - 'valid': Already in correct format and within range
    - 'convertible': Can be fixed (missing day, wrong format) and within range
    - 'invalid': Cannot be converted or outside valid range
    - 'null': NULL/NaN/empty values

    Returns DataFrame with:
    - date_tag: validation tag
    - date_converted: standardized date in YYYY-MM-DD format or None
    """

    default_min_threshold = pd.Timestamp(min_date)
    default_max_threshold = pd.Timestamp(max_date)

    def process_date(row):
        """Process a single date value with per-row thresholds"""

        val = row[column_name]

        # Determine thresholds for this row
        # Use per-row threshold if available and not null, otherwise use default
        if min_date_col and min_date_col in df.columns and pd.notna(row[min_date_col]):
            min_threshold = pd.Timestamp(row[min_date_col])
            # Remove timezone info if present to allow comparison
            if min_threshold.tzinfo is not None:
                min_threshold = min_threshold.tz_localize(None)
        else:
            min_threshold = default_min_threshold

        if max_date_col and max_date_col in df.columns and pd.notna(row[max_date_col]):
            max_threshold = pd.Timestamp(row[max_date_col])
            # Remove timezone info if present to allow comparison
            if max_threshold.tzinfo is not None:
                max_threshold = max_threshold.tz_localize(None)
        else:
            max_threshold = default_max_threshold

        # Handle NULL values
        if pd.isna(val):
            return 'null', None

        date_str = to_date_str(val)

        # Handle empty or 'nan' strings
        if date_str == '' or date_str.lower() == 'nan':
            return 'null', None

        # Check for invalid year pattern (must start with 19 or 20)
        year_match = re.search(r'(\d{4})', date_str)
        if year_match:
            year_str = year_match.group(1)
            first_digit = year_str[0]
            second_digit = year_str[1]

            # Year must start with 1 or 2
            if first_digit not in ['1', '2']:
                return 'invalid', None

            # If starts with 1, second digit must be 9 (1900-1999)
            if first_digit == '1' and second_digit != '9':
                return 'invalid', None

            # If starts with 2, second digit must be 0 (2000-2099)
            if first_digit == '2' and second_digit != '0':
                return 'invalid', None

        # Try to parse and convert the date
        converted_date = None
        needs_conversion = False

        # Pattern 1: YYYY-MM-DD (standard format with various separators)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month, day = match.groups()
                needs_conversion = False  # Reset for each pattern

                # Store original to check if padding needed
                original_month_len = len(month)
                original_day_len = len(day)

                # Pad month and day if needed
                month = month.zfill(2)
                day = day.zfill(2)

                # Check if padding was needed
                if original_month_len == 1 or original_day_len == 1:
                    needs_conversion = True

                # DON'T swap - the input format is already YYYY-MM-DD
                # (We only swap for DD-MM-YYYY format in Pattern 4)

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    # Different separator means needs conversion
                    if sep != '-':
                        needs_conversion = True

                    return 'convertible' if needs_conversion else 'valid', converted_date
                except:
                    return 'invalid', None

        # Pattern 2: YYYY-MM (missing day)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month = match.groups()
                month = month.zfill(2)
                day = '01'  # Default to first day of month

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # Pattern 3: YYYYMMDD (no separator)
        if re.match(r'^\d{8}$', date_str):
            year = date_str[0:4]
            month = date_str[4:6]
            day = date_str[6:8]

            try:
                parsed = pd.Timestamp(f"{year}-{month}-{day}")
                converted_date = f"{year}-{month}-{day}"

                # Check if within valid range
                if parsed < min_threshold or parsed > max_threshold:
                    return 'invalid', None

                return 'convertible', converted_date
            except:
                return 'invalid', None

        # Pattern 4: DD-MM-YYYY or MM-DD-YYYY (need to detect which)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{4}})$'
            match = re.match(pattern, date_str)
            if match:
                part1, part2, year = match.groups()
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                # Determine if DD-MM-YYYY or MM-DD-YYYY
                # If part1 > 12, it must be day, so DD-MM-YYYY
                if int(part1) > 12:
                    day, month = part1, part2
                # If part2 > 12, it must be day, so MM-DD-YYYY
                elif int(part2) > 12:
                    month, day = part1, part2
                # Ambiguous case: assume MM-DD-YYYY (common US format)
                else:
                    # Could be either, default to MM-DD-YYYY but mark as needing review
                    month, day = part1, part2

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # If nothing matched, it's invalid
        return 'invalid', None

    # Apply processing to all rows
    results = df.apply(process_date, axis=1)

    df_result = pd.DataFrame(index=df.index)
    df_result[output_tag_col] = results.apply(lambda x: x[0])
    df_result[output_date_col] = pd.to_datetime(results.apply(lambda x: x[1]))

    return df_result

# loan table preparation
str_columns = ['ee_customer_id', 'ln_loan_id'] 

missing_values = ['<NA>', 'None', 'NaN', 'nan', '', 'null', 'NULL']
dt[str_columns] = dt[str_columns].replace(missing_values, np.nan).astype('string')

localize_date_cols = ['ee_permanent_freeze_date','ee_onboarding_date']
for col in localize_date_cols:
    print(col)
    dt[col] = pd.to_datetime(dt[col]).dt.tz_localize(None)

date_col_format = ['ee_resignation_date']
for date_col in date_col_format:
    dt[date_col] = pd.to_datetime(dt[date_col], format='%Y-%m-%d', errors='coerce')

# repayment table preparation
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')



resignation_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_resignation_date']].first(),
    column_name='ee_resignation_date',
    output_tag_col='ee_resignation_date_tag',
    output_date_col='ee_resignation_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

permanent_freeze_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_permanent_freeze_date']].first(),
    column_name='ee_permanent_freeze_date',
    output_tag_col='ee_permanent_freeze_date_tag',
    output_date_col='ee_permanent_freeze_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

resignation_dates = resignation_dates.merge(
        permanent_freeze_dates['ee_permanent_freeze_date_validated'],
        how='left',
        on='ee_customer_id'
)

resignation_dates['ee_resignation_date_correct'] = resignation_dates.loc[:, 'ee_resignation_date_validated'].fillna(resignation_dates['ee_permanent_freeze_date_validated'])

dt = dt.merge(
    resignation_dates['ee_resignation_date_correct'].reset_index(),
    how='left',on='ee_customer_id'
)

dt_repayments['vendor_id_shifted'] = dt_repayments.groupby('loan_id')[['vendor_id']].shift(-1)
dt_repayments['repaid_date_shifted'] = dt_repayments.groupby('loan_id')[['repaid_date']].shift(-1)

resignation_dates_repayments = (
    dt_repayments[['user_id','loan_id','vendor_id','vendor_id_shifted','repaid_date']]
    .query('vendor_id == "TPSD"')
    .fillna('TPSD')
    .query('vendor_id != vendor_id_shifted & vendor_id_shifted == "TPFP"')
    .assign(resignation_date_fp_new = lambda x: x['repaid_date'] + pd.DateOffset(months=1))
    .groupby('user_id')[['resignation_date_fp_new']]
    .max()
    .reset_index()
)

dt = dt.merge(resignation_dates_repayments, how='left', left_on='ee_customer_id', right_on='user_id')
dt['ee_resignation_date_correct'] = dt['ee_resignation_date_correct'].fillna(dt['resignation_date_fp_new'])

ee_permanent_freeze_date
ee_onboarding_date


In [10]:
dt.head()

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag,ee_resignation_date_correct,user_id,resignation_date_fp_new
0,1041820,helenmaepepito@gmail.com,+63 (905) 088 8902,Helenmae,Macatol,Pepito,1977-11-05,Female,1,1,2,2,None,None,2,Region VII,None,MABOLO,1147 Gil Tudtud 3rd Street,None,None,straight kiwi hotel,None,regular,Married,None,true,2023-05-06 09:34:58,None,Risk,None,None,None,None,associate,Permanent Job (Private sector),None,NaT,NaT,employer,Frozen,183,120,119,127,549,A,10158.000000000,None,Innodata,None,"[""7"",""22""]",1,-1,None,None,3.5,8,IN_PROGRESS,2023-04-20 16:29:28.000000,NaT,2023-04-20 16:29:28+00:00,2025-08-28 02:24:43+00:00,700,14000,13000,1,188,40,1,1,1,1,1,1,1,1,3709537,Tendo,Approved/Disbursed,PH_GCASH,500.0,45.00,3,2025-06-12 04:52:08+00:00,1,2025-09-22,0,0,0,0,2025-07-07,0.0,0.0,0.0,0.000,1,1,1,1,NaT,NaN,NaT
1,1235970,acdalromnick@gmail.com,+63 (938) 735 3743,romnick,cuizon,acdal,1988-08-12,Male,0,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,TUGATOG,127-90 dulong bronze tugatog malabon city,None,None,bronze,None,regular,Single,None,true,2024-07-22 12:00:25,2024-04-03,Risk,None,None,None,None,installer carpenter,Permanent Job (Private sector),None,NaT,NaT,employer,Frozen,183,118,119,117,537,A,10258.000000000,15.000000000,WPI,None,"[""15"",""30""]",1,-1,None,None,4.0,10,PARKED,2024-07-09 14:20:05.000000,NaT,2024-07-09 14:20:05+00:00,2026-02-03 15:34:54+00:00,2000,11500,10500,1,150,30,1,1,1,1,1,1,1,1,2426212,Tendo,Approved/Disbursed,PH_GCASH,500.0,42.00,3,2024-11-14 18:34:34+00:00,1,2025-02-15,0,0,0,0,2024-11-30,0.0,0.0,0.0,0.000,1,1,1,1,NaT,NaN,NaT
2,525180,minettegalang@gmail.com,+63 (968) 601 9712,Minette,Mendoza,Galang,1975-01-12,Female,1,1,2,2,None,None,2,None,None,None,146 7th st. Countryside,None,1608,None,None,None,Married,None,None,2021-08-21 09:14:35,None,Risk,None,"130,Terminated",None,None,Customer Service,Permanent Job (Private sector),None,2022-07-29 06:19:13,2022-02-08,employer,Frozen,123,120,110,127,480,C,10093.000000000,None,MiraMed Global Services,None,"[""15""]",1,14,U1510 One corporate centre julia vargas

In [11]:
# Your modified code
bucket_name = GS_BUCKET
pickle_filename = f"resignation_data_{CURRENT_DATE}"

# Construct the new filename with .pkl extension
data_filename = f"{pickle_filename}.pkl"
print(data_filename)

destination_blob_name = f"{CLOUDPATH}/{data_filename}"

# Save the DataFrame (or any object) as pickle to GCS
save_pickle_to_gcs(dt[['ee_customer_id','ee_resignation_date_correct']], bucket_name, destination_blob_name)



resignation_data_20260325.pkl
Pickle file uploaded to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Tendo_Model_Monitoring_Data/resignation_data_20260325.pkl


## Outstanding balance code

In [12]:
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

# converting loan_id and user_id to str in order to be aligned with loan data
dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')

# merging with loan data to get columns for actual osbal calculations
dt_repayments = dt_repayments.merge(dt[['ln_loan_id','ln_original_principal', 'ln_orig_interest_fees']].drop_duplicates(),
                                    how='left', left_on='loan_id', right_on='ln_loan_id')

# converting dates
dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

# Calculate cumulative paid amount per loan
dt_repayments['cumulative_paid_principal'] = dt_repayments.groupby('loan_id')['paid_principal'].cumsum()

# Calculate osbal_after directly from loan_amount - cumulative_paid
dt_repayments['osbal_after'] = dt_repayments['ln_original_principal'] - dt_repayments['cumulative_paid_principal']

# Calculate osbal_before by shifting osbal_after and using loan_amount for first payment
dt_repayments['osbal_before'] = dt_repayments.groupby('loan_id')['osbal_after'].shift(1)

# Fill first payment osbal_before with loan_amount
mask = dt_repayments['osbal_before'].isna()
dt_repayments.loc[mask, 'osbal_before'] = dt_repayments.loc[mask, 'ln_original_principal']

In [13]:
dt_repayments.sample(2)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD,vendor_id_shifted,repaid_date_shifted,ln_loan_id,ln_original_principal,ln_orig_interest_fees,cumulative_paid_principal,osbal_after,osbal_before
11611195,1364311,4294802,3,TPSD,COPA,2025-11-21,2025-10-30,78.172,15.158,93.33,78.172,15.158,93.33,769.580,22,TPSD,2025-12-03,4294802,1000.0,120.0,230.421,769.579,847.751
4700893,1290503,2329876,1,TPSD,COPA,2024-11-15,2024-11-15,320.183,221.487,541.67,320.183,221.487,541.67,9679.817,0,TPSD,2024-11-29,2329876,10000.0,3000.0,320.183,9679.817,10000.000


**The tables required for this report:**

1.  `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api` -   dt_prod_api
2.  `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table` -   dt_prod_batch
3.  `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`   -  dt_oop_targets 
4.  `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`    -   dt_bs_oop_new
5.  `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`  -   dt_bs_oop_ex
6.  `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`   -   dt_bs_attr   
7.  `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`    -   dt_bs_attr
8.  `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`  -   dt_res

In [15]:
bq = bigquery.Client(PROJECT_ID)

In [16]:
log("  prod API scores (new users)...")
_prod_api_raw = bq.query("""
    SELECT
        employee_id                        AS ee_customer_id,
        run_date,
        ee_attrition_risk_segment          AS attrition_risk_segment,
        ee_attrition_time_to_leave         AS attrition_time_to_leave,
        oop_score                          AS oop_score_prod,
        oop_risk_segment                   AS oop_risk_segment_prod
    FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
""").to_dataframe()

[15:53:35]   prod API scores (new users)...


In [17]:
log("  prod batch scores (existing users)...")
_prod_batch_raw = bq.query("""
    SELECT
        employee_id                        AS ee_customer_id,
        run_date,
        ee_attrition_risk_segment          AS attrition_risk_segment,
        ee_attrition_time_to_leave         AS attrition_time_to_leave,
        oop_score                          AS oop_score_prod,
        oop_risk_segment                   AS oop_risk_segment_prod
    FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
""").to_dataframe()

[15:54:01]   prod batch scores (existing users)...


In [18]:
log("  OOP matured targets...")
dt_oop_targets = bq.query("""
    SELECT user_id AS ee_customer_id, target AS oop_target
    FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
    WHERE target_maturity_flag = 1
""").to_dataframe()
dt_oop_targets["ee_customer_id"] = dt_oop_targets["ee_customer_id"].astype(str)

[15:54:43]   OOP matured targets...


In [19]:
log("  backscored OOP — new users...")
dt_bs_oop_new = bq.query("""
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
""").to_dataframe()
dt_bs_oop_new["ee_customer_id"] = dt_bs_oop_new["ee_customer_id"].astype(str)

[15:55:52]   backscored OOP — new users...


In [20]:
log("  backscored OOP — existing users...")
dt_bs_oop_ex = bq.query("""
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`
""").to_dataframe()
dt_bs_oop_ex["ee_customer_id"] = dt_bs_oop_ex["ee_customer_id"].astype(str)
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = dt_bs_oop_ex["calc_date"] - pd.DateOffset(months=1)
dt_bs_oop_ex["calc_date_ym"] = (
    dt_bs_oop_ex["calc_date_correct"].dt.year * 100
    + dt_bs_oop_ex["calc_date_correct"].dt.month
)

[15:56:52]   backscored OOP — existing users...


In [21]:
log("  backscored attrition...")
dt_bs_attr = bq.query("""
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`
""").to_dataframe()
dt_bs_attr["ee_customer_id"] = dt_bs_attr["ee_customer_id"].astype(str)
dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = dt_bs_attr["calc_date"] - pd.DateOffset(months=1)
dt_bs_attr["calc_date_ym"] = (
    dt_bs_attr["calc_date_correct"].dt.year * 100
    + dt_bs_attr["calc_date_correct"].dt.month
)
dt_bs_attr["is_new_customer_flag_1m"] = (
    (dt_bs_attr["ee_onboarding_month"] == dt_bs_attr["calc_date_ym"]).astype(int)
)

[15:57:43]   backscored attrition...


In [22]:
log("  raw features (GCS)...")
# dt_raw = pd.read_pickle(generate_bucket_url("Oleh/tendo/data/raw_data_14012026.pkl", GS_BUCKET))
dt_raw = dt.copy()
dt_raw["ee_customer_id"] = dt_raw["ee_customer_id"].astype(str)
dt_raw["ee_onboarding_date"] = pd.to_datetime(dt_raw["ee_onboarding_date"]).dt.tz_localize(None)

# Keep only the two columns we need from dt_raw to avoid any future collision
dt_raw_slim = dt_raw[["ee_customer_id", "ee_onboarding_date"]].drop_duplicates()

[16:01:16]   raw features (GCS)...


In [23]:
log("  resignation data (GCS)...")
dt_res = pd.read_pickle(
    generate_bucket_url(f"{CLOUDPATH}/{data_filename}", GS_BUCKET)
)
dt_res["ee_customer_id"] = dt_res["ee_customer_id"].astype(str)

log("All raw data loaded.")

[16:03:55]   resignation data (GCS)...
[16:04:11] All raw data loaded.


In [24]:
dt_res.shape

(4168379, 2)

In [26]:
dt_raw_slim.shape

(1742823, 2)

In [48]:
# =============================================================================
# STEP 2 — BUILD CLEAN BASE FRAMES
# These are the starting points for every section below.
# We never mutate these — each section .copy()s from them.
# =============================================================================

In [27]:
log("=== STEP 2: Building clean base frames ===")

# ── Prod API base ─────────────────────────────────────────────────────────────
# Add date/YM fields. Merge onboarding date once here so it's always present.
prod_api_base = _prod_api_raw.copy()
prod_api_base["ee_customer_id"] = prod_api_base["ee_customer_id"].astype(str)
prod_api_base["run_date"] = pd.to_datetime(prod_api_base["run_date"]).dt.normalize()
prod_api_base["run_date_ym"] = (
    prod_api_base["run_date"].dt.year * 100 + prod_api_base["run_date"].dt.month
)
# Attach onboarding date to the base frame once
prod_api_base = prod_api_base.merge(dt_raw_slim, how="left", on="ee_customer_id")
prod_api_base["ee_onboarding_date"] = pd.to_datetime(
    prod_api_base["ee_onboarding_date"]
).dt.normalize()
prod_api_base["onb_rd_diff"] = (
    abs(prod_api_base["run_date"] - prod_api_base["ee_onboarding_date"])
).dt.days
prod_api_base["onboarding_date_ym"] = (
    prod_api_base["ee_onboarding_date"].dt.year * 100
    + prod_api_base["ee_onboarding_date"].dt.month
)

# ── Prod batch base ───────────────────────────────────────────────────────────
prod_batch_base = _prod_batch_raw.copy()
prod_batch_base["ee_customer_id"] = prod_batch_base["ee_customer_id"].astype(str)
prod_batch_base = prod_batch_base.drop_duplicates(
    subset=["ee_customer_id", "run_date", "attrition_time_to_leave", "oop_score_prod"]
)
prod_batch_base["run_date"] = pd.to_datetime(prod_batch_base["run_date"], errors="coerce")
prod_batch_base["run_date_ym"] = (
    prod_batch_base["run_date"].dt.year * 100 + prod_batch_base["run_date"].dt.month
)
# Attach onboarding date to the base frame once
prod_batch_base = prod_batch_base.merge(dt_raw_slim, how="left", on="ee_customer_id")
prod_batch_base["ee_onboarding_date"] = pd.to_datetime(
    prod_batch_base["ee_onboarding_date"]
).dt.normalize()
prod_batch_base["onboarding_date_ym"] = (
    prod_batch_base["ee_onboarding_date"].dt.year * 100
    + prod_batch_base["ee_onboarding_date"].dt.month
)
prod_batch_base["onb_rd_diff"] = (
    abs(prod_batch_base["run_date"] - prod_batch_base["ee_onboarding_date"])
).dt.days

log("  prod_api_base and prod_batch_base ready.")
log(f"  prod_api_base columns: {list(prod_api_base.columns)}")
log(f"  prod_batch_base columns: {list(prod_batch_base.columns)}")

[16:06:25] === STEP 2: Building clean base frames ===
[16:06:26]   prod_api_base and prod_batch_base ready.
[16:06:26]   prod_api_base columns: ['ee_customer_id', 'run_date', 'attrition_risk_segment', 'attrition_time_to_leave', 'oop_score_prod', 'oop_risk_segment_prod', 'run_date_ym', 'ee_onboarding_date', 'onb_rd_diff', 'onboarding_date_ym']
[16:06:26]   prod_batch_base columns: ['ee_customer_id', 'run_date', 'attrition_risk_segment', 'attrition_time_to_leave', 'oop_score_prod', 'oop_risk_segment_prod', 'run_date_ym', 'ee_onboarding_date', 'onboarding_date_ym', 'onb_rd_diff']


In [49]:
# =============================================================================
# STEP 3 — OOP MASTER FRAMES  (pages 2, 3, 4)
# =============================================================================

In [28]:
log("=== STEP 3: Building OOP master frames ===")

# ── OOP New Users ─────────────────────────────────────────────────────────────
oop_new = (
    prod_api_base.copy()
    .merge(dt_oop_targets, how="left", on="ee_customer_id")
    .merge(
        dt_bs_oop_new[[
            "ee_customer_id", "score_oop",
            "osbal_as_of_resignation_date",
            "osbal_as_of_oop_eligible_date",
            "osbal_as_of_current_date",
        ]],
        how="left", on="ee_customer_id",
    )
)
oop_new["osbal_as_of_oop_eligible_date_log"] = np.log1p(
    oop_new["osbal_as_of_oop_eligible_date"]
)

# One row per customer, closest-to-onboarding observation
oop_new_calc = (
    oop_new.dropna(subset=["ee_onboarding_date", "oop_target"])
    .sort_values(["onb_rd_diff", "run_date"])
    .drop_duplicates(subset=["ee_customer_id"], keep="first")
    .copy()
)
log(f"  oop_new_calc: {len(oop_new_calc):,} rows")

# ── OOP Existing Users ────────────────────────────────────────────────────────
oop_existing = (
    prod_batch_base.copy()
    .merge(dt_oop_targets, how="left", on="ee_customer_id")
    .merge(
        dt_bs_oop_new[[
            "ee_customer_id",
            "osbal_as_of_resignation_date",
            "osbal_as_of_oop_eligible_date",
            "osbal_as_of_current_date",
        ]],
        how="left", on="ee_customer_id",
    )
    .merge(
        dt_bs_oop_ex[["ee_customer_id", "calc_date_ym", "score_oop"]],
        how="left",
        left_on=["ee_customer_id", "run_date_ym"],
        right_on=["ee_customer_id", "calc_date_ym"],
    )
)
oop_existing["osbal_as_of_oop_eligible_date_log"] = np.log1p(
    oop_existing["osbal_as_of_oop_eligible_date"]
)

oop_existing_calc = (
    oop_existing.dropna(subset=["ee_onboarding_date", "oop_target"])
    .sort_values(["ee_customer_id", "run_date"])
    .copy()
)
log(f"  oop_existing_calc: {len(oop_existing_calc):,} rows")

[16:07:24] === STEP 3: Building OOP master frames ===
[16:07:25]   oop_new_calc: 1,182 rows
[16:07:25]   oop_existing_calc: 15,652 rows


In [50]:
# =============================================================================
# STEP 4 — ATTRITION MASTER FRAMES  (pages 2, 5, 6)
# =============================================================================

In [29]:
log("=== STEP 4: Building Attrition master frames ===")

def build_attrition_targets(df, score_date_col):
    """
    Add time_to_attrition, attrition_event, score_attr_corrected to a frame
    that already has ee_resignation_date_correct from dt_res.
    score_date_col is the column to measure 'months until resignation' from.
    """
    df = df.copy()
    resign_dt = pd.to_datetime(df["ee_resignation_date_correct"], errors="coerce")
    score_dt  = pd.to_datetime(df[score_date_col],               errors="coerce")

    months_diff = (
        (resign_dt.dt.year  - score_dt.dt.year)  * 12
      + (resign_dt.dt.month - score_dt.dt.month)
    )
    df["time_to_attrition"]   = np.where(resign_dt.isna(), np.nan, months_diff)
    df["attrition_event"]     = resign_dt.notna().astype(int)
    df["score_attr_corrected"] = df["score_attr"].replace(np.inf, 15)
    return df


# ── Attrition New Users — Prod v1.0 ───────────────────────────────────────────
# Merge chain: prod_api_base → dt_res → dt_bs_attr (matched by run_date_ym)
# dt_res may share column names with prod_api_base — handle via suffix rename
attr_api_tmp = prod_api_base.copy().merge(
    dt_res, how="left", on="ee_customer_id", suffixes=("", "_res")
)
# Drop any _res-suffixed duplicates (dt_res columns that already exist in prod_api_base)
attr_api_tmp = attr_api_tmp[[c for c in attr_api_tmp.columns if not c.endswith("_res")]]

attr_api_tmp = attr_api_tmp.merge(
    dt_bs_attr[[
        "ee_customer_id", "calc_date_ym",
        "score_attr", "score_attr_segment", "is_new_customer_flag_1m",
    ]],
    how="left",
    left_on=["ee_customer_id", "run_date_ym"],
    right_on=["ee_customer_id", "calc_date_ym"],
    suffixes=("", "_bsattr"),
)
attr_api_tmp = attr_api_tmp[[c for c in attr_api_tmp.columns if not c.endswith("_bsattr")]]

# ee_onboarding_date already in prod_api_base — just dropna and dedup
attr_prod_api_calc = (
    attr_api_tmp
    .dropna(subset=["ee_onboarding_date"])   # ← was the KeyError line
    .sort_values(["onb_rd_diff", "run_date"])
    .drop_duplicates(subset=["ee_customer_id"], keep="first")
    .copy()
)
attr_prod_api_calc = build_attrition_targets(attr_prod_api_calc, "run_date")

# Production segment labels
attr_prod_api_calc["attrition_score_prod"] = (
    attr_prod_api_calc["attrition_time_to_leave"]
    .replace("12+", "15")
    .astype("float")
)
attr_prod_api_calc["attrition_risk_segment_prod"] = (
    attr_prod_api_calc["attrition_score_prod"].replace(ATTRITION_SCORE_MAP)
)
log(f"  attr_prod_api_calc (new, prod): {len(attr_prod_api_calc):,} rows")

# ── Attrition New Users — BS v2.0 ─────────────────────────────────────────────
attr_bs_new_tmp = dt_bs_attr.merge(
    dt_res, how="left", on="ee_customer_id", suffixes=("", "_res")
)
attr_bs_new_tmp = attr_bs_new_tmp[[c for c in attr_bs_new_tmp.columns if not c.endswith("_res")]]

attr_bs_new_calc = (
    attr_bs_new_tmp.query("is_new_customer_flag_1m == 1").copy()
)
attr_bs_new_calc = build_attrition_targets(attr_bs_new_calc, "calc_date_correct")
log(f"  attr_bs_new_calc (new, BS): {len(attr_bs_new_calc):,} rows")

# ── Attrition Existing Users — Prod v1.0 ──────────────────────────────────────
attr_batch_tmp = prod_batch_base.copy().merge(
    dt_res, how="left", on="ee_customer_id", suffixes=("", "_res")
)
attr_batch_tmp = attr_batch_tmp[[c for c in attr_batch_tmp.columns if not c.endswith("_res")]]

attr_batch_tmp = attr_batch_tmp.merge(
    dt_bs_attr[[
        "ee_customer_id", "calc_date_ym",
        "score_attr", "score_attr_segment", "is_new_customer_flag_1m",
    ]],
    how="left",
    left_on=["ee_customer_id", "run_date_ym"],
    right_on=["ee_customer_id", "calc_date_ym"],
    suffixes=("", "_bsattr"),
)
attr_batch_tmp = attr_batch_tmp[[c for c in attr_batch_tmp.columns if not c.endswith("_bsattr")]]

attr_prod_batch_calc = (
    attr_batch_tmp
    .dropna(subset=["ee_onboarding_date"])
    .copy()
)
attr_prod_batch_calc = build_attrition_targets(attr_prod_batch_calc, "run_date")
attr_prod_batch_calc["attrition_score_prod"] = (
    attr_prod_batch_calc["attrition_time_to_leave"]
    .replace("12+", "15")
    .astype("float")
)
attr_prod_batch_calc["attrition_risk_segment_prod"] = (
    attr_prod_batch_calc["attrition_score_prod"].replace(ATTRITION_SCORE_MAP)
)
log(f"  attr_prod_batch_calc (existing, prod): {len(attr_prod_batch_calc):,} rows")

# ── Attrition Existing Users — BS v2.0 ────────────────────────────────────────
attr_bs_existing_tmp = dt_bs_attr.merge(
    dt_res, how="left", on="ee_customer_id", suffixes=("", "_res")
)
attr_bs_existing_tmp = attr_bs_existing_tmp[
    [c for c in attr_bs_existing_tmp.columns if not c.endswith("_res")]
]
attr_bs_existing_calc = (
    attr_bs_existing_tmp.query("is_new_customer_flag_3m == 0").copy()
)
attr_bs_existing_calc = build_attrition_targets(attr_bs_existing_calc, "calc_date_correct")
log(f"  attr_bs_existing_calc (existing, BS): {len(attr_bs_existing_calc):,} rows")

log("All master frames ready.")

[16:08:24] === STEP 4: Building Attrition master frames ===
[16:08:27]   attr_prod_api_calc (new, prod): 11,297 rows
[16:08:40]   attr_bs_new_calc (new, BS): 1,694,829 rows
[16:08:51]   attr_prod_batch_calc (existing, prod): 9,857,088 rows
[16:09:11]   attr_bs_existing_calc (existing, BS): 32,183,179 rows
[16:09:11] All master frames ready.


# =============================================================================

# PAGE 2 — MODEL PERFORMANCE: Gini & C-Index

# =============================================================================

In [31]:
log("=== PAGE 2: Model Performance Gini / C-Index ===")

MON_WINDOW = {
    "Jul 2025":      {"start": "2025-07-01", "end": "2025-07-31"},
    "Aug 2025":      {"start": "2025-08-01", "end": "2025-08-31"},
    "Sep 2025":      {"start": "2025-09-01", "end": "2025-09-30"},   
    "Jul-Sep 2025":  {"start": "2025-07-01", "end": "2025-09-30"},
}

def gini_df(data, date_col, score_col, target_col, model, version, user_type, periods, weight_col=None):
    g = calc_gini(data, date_col, score_col, target_col, periods, weight_col)
    g["Model"]        = model
    g["ModelVersion"] = version
    g["UserType"]     = user_type
    return g

pieces = [
    gini_df(oop_new_calc, "ee_onboarding_date", "oop_score_prod", "oop_target",
            "OOP Repay", "v1.0 Prod", "New users (< 3MOB)", MON_WINDOW),
    gini_df(oop_new_calc[oop_new_calc["score_oop"].notna()],
            "ee_onboarding_date", "score_oop", "oop_target",
            "OOP Repay", "v2.0 BS", "New users (< 3MOB)", MON_WINDOW),
    gini_df(oop_existing_calc, "run_date", "oop_score_prod", "oop_target",
            "OOP Repay", "v1.0 Prod", "Existing users (>= 3MOB)", MON_WINDOW),
    gini_df(oop_existing_calc[oop_existing_calc["score_oop"].notna()],
            "run_date", "score_oop", "oop_target",
            "OOP Repay", "v2.0 BS", "Existing users (>= 3MOB)", MON_WINDOW),
    gini_df(attr_prod_api_calc[attr_prod_api_calc["attrition_score_prod"].notna()],
            "run_date", "attrition_score_prod", "attrition_event",
            "Attrition", "v1.0 Prod", "New users (< 3MOB)", MON_WINDOW),
    gini_df(attr_bs_new_calc[attr_bs_new_calc["score_attr_corrected"].notna()],
            "calc_date_correct", "score_attr_corrected", "attrition_event",
            "Attrition", "v2.0 BS", "New users (< 3MOB)", MON_WINDOW),
    gini_df(attr_prod_batch_calc[attr_prod_batch_calc["attrition_score_prod"].notna()],
            "run_date", "attrition_score_prod", "attrition_event",
            "Attrition", "v1.0 Prod", "Existing users (>= 3MOB)", MON_WINDOW),
    gini_df(attr_bs_existing_calc[attr_bs_existing_calc["score_attr_corrected"].notna()],
            "calc_date_correct", "score_attr_corrected", "attrition_event",
            "Attrition", "v2.0 BS", "Existing users (>= 3MOB)", MON_WINDOW),
]

# Fixed Dev numbers from the original slide (do not change month-to-month)
dev_rows = pd.DataFrame([
    {"UserType": "New users (< 3MOB)",       "Model": "OOP Repay",  "ModelVersion": "v1.0 Prod", "Period": "Dev", "Gini": 0.30},
    {"UserType": "Existing users (>= 3MOB)", "Model": "OOP Repay",  "ModelVersion": "v1.0 Prod", "Period": "Dev", "Gini": 0.32},
    {"UserType": "New users (< 3MOB)",       "Model": "Attrition",  "ModelVersion": "v1.0 Prod", "Period": "Dev", "Gini": 0.66},
    {"UserType": "Existing users (>= 3MOB)", "Model": "Attrition",  "ModelVersion": "v1.0 Prod", "Period": "Dev", "Gini": 0.64},
])

page2 = pd.concat(pieces + [dev_rows], ignore_index=True)
page2["MetricName"] = page2["Model"].map({"OOP Repay": "Gini", "Attrition": "C-Index"})
page2["RunMonth"]   = RUN_MONTH
save_csv(page2, "page02_model_performance_gini_cindex.csv")

[16:15:00] === PAGE 2: Model Performance Gini / C-Index ===
[16:15:30]   ✓ page02_model_performance_gini_cindex.csv  (36 rows, 11 cols)


In [52]:
page2

,Period,Start_Date,End_Date,Sample_Size,Bad_Rate_Pct,Gini,Model,ModelVersion,UserType,MetricName,RunMonth
0,Jul 2025,2025-07-01,2025-07-31,114.0,66.67,0.0017,OOP Repay,v1.0 Prod,New users (< 3MOB),Gini,2026-03
1,Aug 2025,2025-08-01,2025-08-31,77.0,68.83,0.0165,OOP Repay,v1.0 Prod,New users (< 3MOB),Gini,2026-03
2,Sep 2025,2025-09-01,2025-09-30,103.0,71.84,-0.2330,OOP Repay,v1.0 Prod,New users (< 3MOB),Gini,2026-03
3,Jul-Sep 2025,2025-07-01,2025-09-30,294.0,69.05,-0.0752,OOP Repay,v1.0 Prod,New users (< 3MOB),Gini,2026-03
4,Jul 2025,2025-07-01,2025-07-31,107.0,66.36,0.2919,OOP Repay,v2.0 BS,New users (< 3MOB),Gini,2026-03
5,Aug 2025,2025-08-01,2025-08-31,72.0,68.06,-0.0568,OOP Repay,v2.0 BS,New users (< 3MOB),Gini,2026-03
6,Sep 2025,2025-09-01,2025-09-30,98.0,72.45,-0.1784,OOP Repay,v2.0 BS,New users (< 3MOB),Gini,2026-03
7,Jul-Sep 2025,2025-07-01,2025-09-30,277.0,68.95,0.0203,OOP Repay,v2.0 BS,New users (< 3MOB),Gini,2026-03
8,Jul 2025,2025-07-01,2025-07-31,3314.0,72.39,0.2688,OOP Repay,v1.0 Prod,Existing users (>= 3MOB),Gini,2026-03
9,Aug 2025,2025-08-01,2025-08-31,2622.0,70.10,0.2550,OOP Repay,v1.0 Prod,Existing users (>= 3MOB),Gini,2026-03


# =============================================================================
# PAGE 3 — OOP Rank Order: New Users
# =============================================================================

In [33]:
log("=== PAGE 3: OOP Rank Order — New Users ===")

def derive_bs_segment_labels(df, score_col, prod_segment_col):
    """
    Re-derive production segment bin edges from observed prod scores,
    then cut the backscored score into the same A-F segments.
    Mirrors the bin-derivation logic in the original notebook exactly.
    """
    valid = df[df[prod_segment_col].notna() & df[score_col].notna()].copy()
    valid["_seg"]   = valid[prod_segment_col].astype(str)
    valid["_score"] = pd.to_numeric(valid[score_col], errors="coerce")

    mm = (
        valid.groupby("_seg")["_score"]
        .agg(["min", "max"])
        .reindex(list("ABCDEF"))
        .clip(0, 1)
        .interpolate(limit_direction="both")
    )
    order = list("ABCDEF")
    cuts = pd.Series(
        [(mm.loc[h, "min"] + mm.loc[l, "max"]) / 2
         for h, l in zip(order[:-1], order[1:])],
        index=[f"{h}/{l}" for h, l in zip(order[:-1], order[1:])]
    ).cummin()
    edges  = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
    labels = list("FEDCBA")
    return pd.cut(df[score_col], bins=edges, labels=labels,
                  right=False, include_lowest=True)


def build_oop_pivot(df, segment_col, filter_expr, model_label, period_label):
    """Compute the OOP rank-order pivot table for one model/period."""
    sub = df.query(filter_expr).copy() if filter_expr else df.copy()
    sub["oop_target_bad"]    = 1 - sub["oop_target"]
    sub["osbal_current_bad"] = sub["oop_target_bad"] * sub["osbal_as_of_current_date"]

    pt = (
        sub.groupby(segment_col, observed=True)
        .agg(
            Count                  = ("ee_customer_id", "count"),
            Bad_Count              = ("oop_target_bad", "sum"),
            Sum_Osbal_Resignation  = ("osbal_as_of_resignation_date", "sum"),
            Sum_Osbal_Bads_Today   = ("osbal_current_bad", "sum"),
        )
        .reset_index()
        .rename(columns={segment_col: "RiskGroup"})
    )
    pt["Bad_Rate_Pct"]         = (pt["Bad_Count"] / pt["Count"] * 100).round(1)
    pt["Outstanding_Bad_Rate"] = (
        pt["Sum_Osbal_Bads_Today"] / pt["Sum_Osbal_Resignation"] * 100
    ).round(1)
    pt["ModelVersion"] = model_label
    pt["Period"]       = period_label
    pt["Segment_Order"] = pt["RiskGroup"].astype(str).map(OOP_SEGMENT_ORDER)
    pt["RunMonth"]      = RUN_MONTH
    return pt.drop(columns=["Bad_Count"])


# Derive BS segment labels for new users using prod boundaries
oop_new_calc["oop_risk_segment_bs"] = derive_bs_segment_labels(
    oop_new_calc, "score_oop", "oop_risk_segment_prod"
)

p3_v1 = build_oop_pivot(
    oop_new_calc, "oop_risk_segment_prod",
    "onboarding_date_ym >= 202507 & onboarding_date_ym <= 202509",
    "OOP v1.0 Prod", "Jul-Sep 2025"
)
p3_v2 = build_oop_pivot(
    oop_new_calc[oop_new_calc["score_oop"].notna()],
    "oop_risk_segment_bs",
    "onboarding_date_ym >= 202507 & onboarding_date_ym <= 202509",
    "OOP v2.0 BS", "Jul-Sep 2025"
)
page3 = pd.concat([p3_v1, p3_v2], ignore_index=True).sort_values(
    ["ModelVersion", "Segment_Order"]
)
save_csv(page3, "page03_oop_rank_order_new_users.csv")

[16:31:10] === PAGE 3: OOP Rank Order — New Users ===
[16:31:10]   ✓ page03_oop_rank_order_new_users.csv  (12 rows, 10 cols)


# =============================================================================
# PAGE 4 — OOP Rank Order: Existing Users
# =============================================================================

In [34]:
log("=== PAGE 4: OOP Rank Order — Existing Users ===")

oop_existing_calc["oop_risk_segment_bs"] = derive_bs_segment_labels(
    oop_existing_calc, "score_oop", "oop_risk_segment_prod"
)

p4_v1 = build_oop_pivot(
    oop_existing_calc, "oop_risk_segment_prod",
    "run_date_ym == 202507",
    "OOP v1.0 Prod", "Jul 2025"
)
p4_v2 = build_oop_pivot(
    oop_existing_calc[oop_existing_calc["score_oop"].notna()],
    "oop_risk_segment_bs",
    "run_date_ym == 202507",
    "OOP v2.0 BS", "Jul 2025"
)
page4 = pd.concat([p4_v1, p4_v2], ignore_index=True).sort_values(
    ["ModelVersion", "Segment_Order"]
)
save_csv(page4, "page04_oop_rank_order_existing_users.csv")

[16:32:22] === PAGE 4: OOP Rank Order — Existing Users ===
[16:32:22]   ✓ page04_oop_rank_order_existing_users.csv  (12 rows, 10 cols)


# =============================================================================
# PAGE 5 — Attrition Rank Order: New Users
# =============================================================================

In [35]:
log("=== PAGE 5: Attrition Rank Order — New Users ===")

def build_attrition_pivot(df, segment_col, filter_expr, model_label, period_label):
    sub = df.query(filter_expr).copy() if filter_expr else df.copy()
    pt = (
        sub.groupby(segment_col, observed=True)
        .agg(
            Count            = ("ee_customer_id", "count"),
            Attrition_Rate   = ("attrition_event", "mean"),
            Expected_Avg_TTA = ("score_attr_corrected", "mean"),
            Actual_Avg_TTA   = ("time_to_attrition", "mean"),
        )
        .reset_index()
        .rename(columns={segment_col: "AttritionGroup"})
    )
    pt["Attrition_Rate_Pct"]  = (pt["Attrition_Rate"] * 100).round(1)
    pt["Expected_Avg_TTA"]    = pt["Expected_Avg_TTA"].round(1)
    pt["Actual_Avg_TTA"]      = pt["Actual_Avg_TTA"].round(1)
    pt["ModelVersion"]         = model_label
    pt["Period"]               = period_label
    pt["Segment_Order"]        = pt["AttritionGroup"].map(ATTRITION_ORDER)
    pt["RunMonth"]             = RUN_MONTH
    return pt.drop(columns=["Attrition_Rate"])


# Prod v1.0 — prod segment (attrition_risk_segment_prod), Jun-Aug onboarding
p5_v1 = build_attrition_pivot(
    attr_prod_api_calc, "attrition_risk_segment_prod",
    "onboarding_date_ym >= 202507 & onboarding_date_ym <= 202509",
    "Attrition v1.0 Prod", "Jul-Sep 2025"
)

# BS v2.0 — BS segment (score_attr_segment), Jun-Aug onboarding month
p5_v2 = build_attrition_pivot(
    attr_bs_new_calc, "score_attr_segment",
    "ee_onboarding_month >= 202507 & ee_onboarding_month <= 202509",
    "Attrition v2.0 BS", "Jul-Sep 2025"
)

page5 = pd.concat([p5_v1, p5_v2], ignore_index=True).sort_values(
    ["ModelVersion", "Segment_Order"]
)
save_csv(page5, "page05_attrition_rank_order_new_users.csv")

[16:34:01] === PAGE 5: Attrition Rank Order — New Users ===
[16:34:01]   ✓ page05_attrition_rank_order_new_users.csv  (10 rows, 9 cols)


# =============================================================================
# PAGE 6 — Attrition Rank Order: Existing Users
# =============================================================================

In [36]:
log("=== PAGE 6: Attrition Rank Order — Existing Users ===")

# Prod v1.0 — Jul 2025 snapshot
p6_v1 = build_attrition_pivot(
    attr_prod_batch_calc.query("run_date_ym == 202507"),
    "attrition_risk_segment_prod", None,
    "Attrition v1.0 Prod", "Jul 2025"
)

# BS v2.0 — Jul 2025 snapshot (existing = is_new_customer_flag_3m == 0)
p6_v2 = build_attrition_pivot(
    attr_bs_existing_calc.query("calc_date_ym == 202507"),
    "score_attr_segment", None,
    "Attrition v2.0 BS", "Jul 2025"
)

page6 = pd.concat([p6_v1, p6_v2], ignore_index=True).sort_values(
    ["ModelVersion", "Segment_Order"]
)
save_csv(page6, "page06_attrition_rank_order_existing_users.csv")


[16:34:53] === PAGE 6: Attrition Rank Order — Existing Users ===
[16:34:54]   ✓ page06_attrition_rank_order_existing_users.csv  (10 rows, 9 cols)


In [43]:
# =============================================================================
# PAGES 7–11 — Pure pandas, built entirely from dataframes already in memory.
# No extra BigQuery table names needed beyond what was loaded in Step 1.
# =============================================================================

In [39]:
log("=== PAGE 7: Unit Bad Rate MoM ===")

[17:21:44] === PAGE 7: Unit Bad Rate MoM ===


In [44]:
# ---------------------------------------------------------------------------
# PAGE 7 — Unit Bad Rate MoM
# ---------------------------------------------------------------------------
# Source: oop_new_calc — already has oop_target, osbal columns,
# ee_resignation_date_correct (via dt_res merged in Step 4 attrition build),
# ee_onboarding_date, onboarding_date_ym.
#
# Scorecard version is inferred from onboarding month:
#   SC 1.0 live Jun 2025, SC 2.0 live Nov 2025 — adjust SC2_LIVE_YM if needed.
# ---------------------------------------------------------------------------

In [40]:
SC2_LIVE_YM = 202511   # ← update if SC 2.0 go-live month changes
 
p7 = oop_new_calc.copy()
 
# Scorecard label from onboarding month
p7["Scorecard"] = np.where(p7["onboarding_date_ym"] >= SC2_LIVE_YM, "SC 2.0", "SC 1.0")
p7["OnboardingMonth"] = pd.to_datetime(p7["ee_onboarding_date"]).dt.strftime("%Y-%m")
 
# Resignation-date join already done via dt_res in the attrition master build.
# oop_new_calc was built from oop_new which came from prod_api_base (no dt_res).
# We need ee_resignation_date_correct here — merge it in now.
p7 = p7.merge(
    dt_res[["ee_customer_id", "ee_resignation_date_correct"]],
    how="left", on="ee_customer_id", suffixes=("", "_res")
)
# If the column already existed (from a prior merge), keep the left-table version
if "ee_resignation_date_correct_res" in p7.columns:
    p7["ee_resignation_date_correct"] = p7["ee_resignation_date_correct"].fillna(
        p7["ee_resignation_date_correct_res"]
    )
    p7 = p7.drop(columns=["ee_resignation_date_correct_res"])
 
resign_dt = pd.to_datetime(p7["ee_resignation_date_correct"], errors="coerce")
p7["left_job"]           = (resign_dt <= pd.Timestamp("2026-01-31")).astype(int)
p7["had_oop_outstanding"] = (p7["osbal_as_of_resignation_date"].fillna(0) > 0).astype(int)
p7["oop_flag_bad"]        = np.where(
    (p7["left_job"] == 1) & (p7["had_oop_outstanding"] == 1) & (p7["oop_target"] == 0),
    1, 0
)
p7["cnt_had_oop_if_left"] = p7["left_job"] * p7["had_oop_outstanding"]
 
page7 = (
    p7.groupby(["Scorecard", "OnboardingMonth"], sort=True)
    .agg(
        CntOnboarded         = ("ee_customer_id", "count"),
        CntLeftJob           = ("left_job", "sum"),
        CntHadOopOutstanding = ("cnt_had_oop_if_left", "sum"),
        CntOopFlagBad        = ("oop_flag_bad", "sum"),
    )
    .reset_index()
)
page7["UnitBadRate"]     = page7["CntOopFlagBad"] / page7["CntLeftJob"].replace(0, np.nan)
page7["UnitBadRate_Pct"] = (page7["UnitBadRate"] * 100).round(2)
page7["RunMonth"]        = RUN_MONTH
save_csv(page7, "page07_unit_bad_rate_mom.csv")

[17:22:30]   ✓ page07_unit_bad_rate_mom.csv  (62 rows, 9 cols)


# ---------------------------------------------------------------------------
# PAGE 8 — Peso Bad Rate MoM
# ---------------------------------------------------------------------------
# Same base as page 7 but sums PHP outstanding balances.
# ---------------------------------------------------------------------------

In [41]:
log("=== PAGE 8: Peso Bad Rate MoM ===")
 
p8 = p7.copy()   # already has left_job, oop_target, resign flags
 
p8["os_at_resignation_if_left"] = np.where(
    p8["left_job"] == 1, p8["osbal_as_of_resignation_date"].fillna(0), 0
)
p8["os_at_eligible_if_left"] = np.where(
    p8["left_job"] == 1, p8["osbal_as_of_oop_eligible_date"].fillna(0), 0
)
p8["os_current_if_bad"] = np.where(
    (p8["left_job"] == 1) & (p8["oop_target"] == 0),
    p8["osbal_as_of_current_date"].fillna(0), 0
)
 
page8 = (
    p8.groupby(["Scorecard", "OnboardingMonth"], sort=True)
    .agg(
        TotOsAsOfResignation  = ("os_at_resignation_if_left", "sum"),
        TotOsAsOfOopEligible  = ("os_at_eligible_if_left", "sum"),
        TotOsFromBadCustomers = ("os_current_if_bad", "sum"),
    )
    .reset_index()
)
page8["PesoBadRate"]     = (
    page8["TotOsFromBadCustomers"]
    / page8["TotOsAsOfOopEligible"].replace(0, np.nan)
)
page8["PesoBadRate_Pct"] = (page8["PesoBadRate"] * 100).round(2)
page8["RunMonth"]        = RUN_MONTH
save_csv(page8, "page08_peso_bad_rate_mom.csv")

[17:23:10] === PAGE 8: Peso Bad Rate MoM ===
[17:23:10]   ✓ page08_peso_bad_rate_mom.csv  (62 rows, 8 cols)


# ---------------------------------------------------------------------------
# PAGE 9 — CL+ Incremental Portfolio Exposure
# ---------------------------------------------------------------------------

In [42]:
log("=== PAGE 9: CL+ Exposure ===")
 
CL_PLUS_LAUNCH_YM = 202506   # month CL+ program started (June 2025)
 
# Check if dt_raw already has a credit_limit column
if "credit_limit" in dt_raw.columns:
    log("  Using credit_limit from dt_raw")
    cl_src = dt_raw[["ee_customer_id", "ee_onboarding_date", "credit_limit"]].copy()
    cl_src["ee_customer_id"] = cl_src["ee_customer_id"].astype(str)
 
    # For existing users: those onboarded before CL+ launch
    pre_launch = cl_src[
        pd.to_datetime(cl_src["ee_onboarding_date"]).dt.year * 100
        + pd.to_datetime(cl_src["ee_onboarding_date"]).dt.month < CL_PLUS_LAUNCH_YM
    ].copy()
 
    # old_cl = credit limit at onboarding (first row per customer in dt_raw is the snapshot)
    old_cl = pre_launch.groupby("ee_customer_id")["credit_limit"].first().rename("old_cl")
 
    # new_cl = credit limit after CL+ — use prod_batch_base latest score month as proxy
    # Since the batch table doesn't have credit_limit directly, we approximate:
    # new_cl from dt_raw for customers whose CL changed post-launch would need a
    # separate snapshot.  For now, use dt_raw credit_limit as new_cl too and
    # compare to a separate CL+ delta table if available.
    new_cl = pre_launch.groupby("ee_customer_id")["credit_limit"].last().rename("new_cl")
 
else:
    log("  credit_limit not in dt_raw — fetching from BigQuery (minimal query)")
    _cl_df = bq.query("""
        SELECT
            employee_id  AS ee_customer_id,
            credit_limit,
            onboarding_date
        FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
        WHERE employee_id IS NOT NULL
    """).to_dataframe()
    _cl_df["ee_customer_id"] = _cl_df["ee_customer_id"].astype(str)
    _cl_df["onb_ym"] = (
        pd.to_datetime(_cl_df["onboarding_date"]).dt.year * 100
        + pd.to_datetime(_cl_df["onboarding_date"]).dt.month
    )
 
    pre_launch = _cl_df[_cl_df["onb_ym"] < CL_PLUS_LAUNCH_YM].copy()
    old_cl = pre_launch.groupby("ee_customer_id")["credit_limit"].first().rename("old_cl")
 
    # new_cl: latest credit limit from prod batch (most recent run_date per customer)
    _latest_batch = (
        prod_batch_base.sort_values("run_date")
        .groupby("ee_customer_id").last()
        .reset_index()
    )
    if "credit_limit" in _latest_batch.columns:
        new_cl = _latest_batch.set_index("ee_customer_id")["credit_limit"].rename("new_cl")
    else:
        # Fetch new CL from API table using latest run date
        _new_cl_df = bq.query("""
            SELECT employee_id AS ee_customer_id, credit_limit AS new_cl
            FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
            WHERE run_date = (SELECT MAX(run_date)
                              FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`)
        """).to_dataframe()
        _new_cl_df["ee_customer_id"] = _new_cl_df["ee_customer_id"].astype(str)
        new_cl = _new_cl_df.set_index("ee_customer_id")["new_cl"]
 
cl_delta = pd.concat([old_cl, new_cl], axis=1).dropna()
cl_delta = cl_delta[cl_delta.index.isin(old_cl.index)]   # existing users only
cl_delta["cl_delta"] = cl_delta["new_cl"] - cl_delta["old_cl"]
cl_delta["CLCategory"] = np.select(
    [cl_delta["cl_delta"] > 0, cl_delta["cl_delta"] < 0],
    ["CL Increased",           "CL Decreased"],
    default="No CL Increase"
)
cl_delta = cl_delta.reset_index()
 
page9 = (
    cl_delta.groupby("CLCategory")
    .agg(
        UserCount = ("ee_customer_id", "count"),
        SumOldCL  = ("old_cl", "sum"),
        SumNewCL  = ("new_cl", "sum"),
        CLDelta   = ("cl_delta", "sum"),
    )
    .reset_index()
)
page9["PctCLIncrease"]     = page9["CLDelta"] / page9["SumOldCL"].replace(0, np.nan)
page9["PctCLIncrease_Pct"] = (page9["PctCLIncrease"] * 100).round(1)
page9["RunMonth"]          = RUN_MONTH
save_csv(page9, "page09_cl_plus_exposure.csv")

[17:23:50] === PAGE 9: CL+ Exposure ===
[17:23:50]   credit_limit not in dt_raw — fetching from BigQuery (minimal query)


BadRequest: 400 Unrecognized name: credit_limit at [4:13]; reason: invalidQuery, location: query, message: Unrecognized name: credit_limit at [4:13]

Location: asia-southeast1
Job ID: 4624897d-81ae-4951-bcc2-550bed46e3e0
